# M3 — 02 Update DQ Log (manual)

Run **only after Notebook 01 finishes successfully with `ALLOW_WRITES=True`**. Use the **same `run_date`**. Reads staged metrics and MERGEs into the existing `atliq.silver.dq_log`. Does not rebuild the log or automate anything.

**Important:** The DQ counts describe the current processed inputs. For full-refresh tables they describe the entire Bronze snapshot, not just new changes on `run_date`.


In [0]:
# Create run_date widget
dbutils.widgets.text("run_date", "2026-11-01", "Run Date (YYYY-MM-DD)")

# Get selected run date
run_date = dbutils.widgets.get("run_date")

print(f"Pipeline Run Date: {run_date}")

In [0]:
from datetime import date
from pyspark.sql import functions as F
from delta.tables import DeltaTable

SILVER = "atliq.silver"
STAGE_TABLE = f"{SILVER}.dq_stage"
DQ_TABLE = f"{SILVER}.dq_log"

dbutils.widgets.text("run_date", "")
run_date = dbutils.widgets.get("run_date").strip()
assert run_date, "Type a date (YYYY-MM-DD) in the run_date box at the top, then Run all again"
date.fromisoformat(run_date)

assert spark.catalog.tableExists(DQ_TABLE), f"Missing {DQ_TABLE}; do not create/overwrite existing log automatically"
expected = {"run_date", "table_name", "rule", "rows_affected"}
actual = set(spark.table(DQ_TABLE).columns)
assert actual == expected, f"dq_log columns differ: {actual}; expected {expected}. Inspect before MERGE."
assert spark.table(DQ_TABLE).schema["run_date"].dataType.simpleString() in ("string", "date"), "Unsupported dq_log run_date type"
stage = spark.table(STAGE_TABLE).filter(F.col("run_date") == run_date)
n = stage.count()
assert n > 0, f"No counts saved for {run_date}. Run notebook 01 first, with the same run_date"
assert stage.groupBy("table_name", "rule").count().filter("count > 1").count() == 0, "Duplicate staged rule keys"
print("counts to log for", run_date, ":", n)
display(stage.orderBy("table_name", "rule"))

In [0]:
# MERGE one metric per run_date + table + rule; repeated run updates, not duplicates.
run_date_type = spark.table(DQ_TABLE).schema["run_date"].dataType
stage_for_merge = stage.withColumn("run_date", F.col("run_date").cast(run_date_type))
assert spark.table(DQ_TABLE).groupBy("run_date", "table_name", "rule").count().filter("count > 1").count() == 0,     "Existing dq_log has duplicate keys; resolve before MERGE"
(DeltaTable.forName(spark, DQ_TABLE).alias("t")
    .merge(stage_for_merge.alias("s"),
           "t.run_date = s.run_date AND t.table_name = s.table_name AND t.rule = s.rule")
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute())

log = spark.table(DQ_TABLE).filter(F.col("run_date") == run_date)
print("dq_log rows for", run_date, ":", log.count())
dups = log.groupBy("table_name", "rule").count().filter("count > 1").count()
print("duplicate rules:", dups)
display(log.orderBy("table_name", "rule"))